In [ ]:
# Celda 1
# Paths and parameters:
%cd ..

AYTO_ZIP = "sources/ayto_alturas/ALTURAS_EDIFICIOS.ZIP"
GPKG = "data/catastro_manzana.gpkg"
MESHES = "data/lod2_meshes.json"
CLOUD = "data/nube_manzana_2023.laz"
OUT_GPKG = "data/validacion.gpkg"

ORIGIN = (440000.0, 4473700.0, 640.0)   # Local origin of the Blender model
CONJUNTO = ["0139402VK4703G", "0139423VK4703G", "0139424VK4703G"]
INNER_MARGIN = 0.5   # Meters removed from the municipal polygon edge, to avoid the walls of the neighbor
MIN_VALUES = 10     # Fewer cloud points or model vertices than this inside a polygon -> not compared

In [ ]:
# Celda 2
import json
import numpy as np
import laspy
import shapely
import geopandas as gpd

In [ ]:
# Celda 3
# Municipal height polygons that fall inside the block (any parcel):
parcels = gpd.read_file(GPKG, layer="parcelas")
ayto = gpd.read_file("zip://" + AYTO_ZIP, bbox=tuple(parcels.total_bounds))
ayto = ayto.set_geometry(shapely.force_2d(ayto.geometry))
ayto["centroid"] = ayto.geometry.representative_point()

# Parcel of each polygon = parcel that contains a point inside it:
joined = gpd.sjoin(ayto.set_geometry("centroid"), parcels[["nationalCadastralReference", "geometry"]], predicate="within")
ayto = ayto.loc[joined.index].copy()
ayto["parcel"] = joined["nationalCadastralReference"]
ayto["group"] = np.where(ayto["parcel"].isin(CONJUNTO), "conjunto", "contexto")
ayto = ayto.set_geometry("geometry").drop(columns="centroid")
print(len(ayto), "municipal polygons in the block")
print(ayto["group"].value_counts())

In [ ]:
# Celda 4
# Roof vertices of the LOD2 meshes, in real coordinates.
# Roof = vertices of triangles (walls are quads) that are above the bottom of the mesh.
with open(MESHES, encoding="utf-8") as f:
    meshes = json.load(f)["parts"]

x0, y0, z0 = ORIGIN
rx = []
ry = []
rz = []
for mesh in meshes:
    if mesh["lod"] != 2:
        continue
    lowest = min(v[2] for v in mesh["vertices"])
    for face in mesh["faces"]:
        if len(face) != 3:
            continue
        for index in face:
            x, y, z = mesh["vertices"][index]
            if z > lowest + 0.01:
                rx.append(x + x0)
                ry.append(y + y0)
                rz.append(z + z0)
rx = np.array(rx)
ry = np.array(ry)
rz = np.array(rz)
print(len(rz), "roof vertices (LOD2)")

In [ ]:
# Celda 5
# Building points of the cloud (the input of the model), to compare the data too:
cloud = laspy.read(CLOUD)
is_building = np.array(cloud.classification) == 6
bx = np.array(cloud.x)[is_building]
by = np.array(cloud.y)[is_building]
bz = np.array(cloud.z)[is_building]

# Helper: values of (px, py, pz) inside a polygon:
def values_inside(polygon, px, py, pz):
    xmin, ymin, xmax, ymax = polygon.bounds
    near = (px >= xmin) & (px <= xmax) & (py >= ymin) & (py <= ymax)
    inside = shapely.contains_xy(polygon, px[near], py[near])
    return pz[near][inside]

In [ ]:
# Celda 6
# For each municipal polygon:
#   cloud_z = median of the cloud building points inside it;
#   model_z = median of the LOD2 roof vertices inside it, or (LOD1 or too few vertices)
#             the top of the Catastro part that contains the polygon.
parts = gpd.read_file(GPKG, layer="building_parts_alturas")
parts = parts[parts["numberOfFloorsAboveGround"] > 0]

cloud_z = []
model_z = []
model_source = []
for polygon in ayto.geometry:
    inner = polygon.buffer(-INNER_MARGIN)
    if inner.is_empty:
        inner = polygon

    points = values_inside(inner, bx, by, bz)
    if len(points) >= MIN_VALUES:
        cloud_z.append(np.median(points))
    else:
        cloud_z.append(np.nan)

    roof = values_inside(inner, rx, ry, rz)
    if len(roof) >= MIN_VALUES:
        model_z.append(np.median(roof))
        model_source.append("LOD2")
        continue
    inside_part = parts[parts.geometry.contains(polygon.representative_point())]
    if len(inside_part) > 0:
        row = inside_part.iloc[0]
        model_z.append(row["z_base"] + row["height_m"])
        model_source.append("LOD1")
    else:
        model_z.append(np.nan)
        model_source.append("none")

ayto["cloud_z"] = np.round(cloud_z, 2)
ayto["model_z"] = np.round(model_z, 2)
ayto["model_source"] = model_source
ayto["diff_cloud"] = (ayto["cloud_z"] - ayto["Z_CAMB_ALT"]).round(2)
ayto["diff_model"] = (ayto["model_z"] - ayto["Z_CAMB_ALT"]).round(2)
print(ayto["model_source"].value_counts())

In [ ]:
# Celda 7
# Summary of the differences with the municipal height (Z_CAMB_ALT), in meters:
def summary(label, d):
    d = d.dropna()
    if len(d) == 0:
        print("  ", label, "| n = 0")
        return
    print("  ", label, "| n =", len(d), "| median", round(d.median(), 2), "| median abs", round(d.abs().median(), 2),
          "| within 1 m:", round(100 * (d.abs() <= 1).mean()), "% | within 3 m:", round(100 * (d.abs() <= 3).mean()), "%")

for group in ["conjunto", "contexto"]:
    sub = ayto[ayto["group"] == group]
    print(group)
    summary("cloud      ", sub["diff_cloud"])
    summary("model LOD2 ", sub[sub["model_source"] == "LOD2"]["diff_model"])
    summary("model LOD1 ", sub[sub["model_source"] == "LOD1"]["diff_model"])

In [ ]:
# Celda 8
# Largest model differences in the conjunto:
columns = ["parcel", "FECHA_ALTA", "Z_CAMB_ALT", "cloud_z", "model_z", "model_source", "diff_cloud", "diff_model", "area_m2"]
ayto["area_m2"] = ayto.geometry.area.round(1)
conjunto = ayto[ayto["group"] == "conjunto"]
conjunto.reindex(conjunto["diff_model"].abs().sort_values(ascending=False).index)[columns].head(10)

In [ ]:
# Celda 9
# Save the comparison (to look at it in QGIS):
ayto.to_file(OUT_GPKG, layer="alturas_ayto_vs_modelo", driver="GPKG")
print("Saved:", OUT_GPKG)